# tpu-retrieval-bench on Colab TPU

Runtime > Change runtime type > TPU v5e-1. Results land in `results/` and are zipped at the end.

In [ ]:
import sys
print(sys.version)
if sys.version_info < (3, 12):
    print('WARNING: python < 3.12. Fallback: pip install "jax[tpu]<0.11" and '
          'pip install --ignore-requires-python -e ".[data,plot]" (repo declares >= 3.12).')

In [ ]:
!git clone https://github.com/maxpetrusenko/tpu-retrieval-bench
%cd tpu-retrieval-bench
!pip install -q -e ".[data,plot]" "jax[tpu]==0.12.0.1" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
# jax pinned to the uv.lock version (0.12.0.1). If pip changed jax, restart the runtime
# (Runtime > Restart session) before running the next cell.

In [ ]:
import jax
dev = jax.devices()[0]
assert dev.platform == "tpu", f"not a TPU runtime: {jax.devices()}"
print(dev.device_kind, len(jax.devices()), "device(s)")

## Synthetic sweep

In [ ]:
!python bench/run.py --algos jit_topk,blockwise,pallas_score,approx_max_k,score_only,select_only --m 1,16,128,1024 --n 65536,1000000 --d 128 --k 10 --dtype bf16

## ann-benchmarks SIFT (525 MB)

In [ ]:
!python bench/download.py sift

In [ ]:
!python bench/run.py --dataset sift-128-euclidean --algos jit_topk,blockwise,pallas_score,approx_max_k,score_only,select_only --k 10 --dtype bf16 --n-test 1000 --batch-size 1000

## Profiler trace for one config

In [ ]:
import numpy as np
import jax.numpy as jnp
from trbench.jit_topk import make_jit_topk
rng = np.random.default_rng(0)
q = jnp.asarray(rng.standard_normal((128, 128), dtype=np.float32))
e = jnp.asarray(rng.standard_normal((1_000_000, 128), dtype=np.float32))
f = make_jit_topk(10)
jax.block_until_ready(f(q, e))  # warm
with jax.profiler.trace("results/profile_jit_topk_m128_n1e6_d128"):
    jax.block_until_ready(f(q, e))

## Plots and download

In [ ]:
!python bench/plot.py

In [ ]:
!zip -qr results.zip results docs/img
from google.colab import files
files.download("results.zip")